In [ ]:
import pandas as pd
import numpy as np

def process_transport_data(stops_path, routes_path, trips_path, stop_times_path):
    print("Dosyalar okunuyor...")
    
    # HATA ÇÖZÜMÜ: Encoding parametresi eklendi ('windows-1254' Türkçe karakterler içindir)
    # Eğer yine hata verirse 'iso-8859-9' veya 'utf-8-sig' deneyebilirsin.
    encoding_type = 'windows-1254' 
    
    try:
        stops = pd.read_csv(stops_path, encoding=encoding_type, sep=',') # Bazen noktalı virgül de olabilir, garanti olsun diye sep eklenebilir ama standart csv için virgül varsayılır.
        routes = pd.read_csv(routes_path, encoding=encoding_type)
        trips = pd.read_csv(trips_path, encoding=encoding_type)
        stop_times = pd.read_csv(stop_times_path, encoding=encoding_type)
    except UnicodeDecodeError:
        print("Windows formatı uymadı, ISO formatı deneniyor...")
        encoding_type = 'iso-8859-9'
        stops = pd.read_csv(stops_path, encoding=encoding_type)
        routes = pd.read_csv(routes_path, encoding=encoding_type)
        trips = pd.read_csv(trips_path, encoding=encoding_type)
        stop_times = pd.read_csv(stop_times_path, encoding=encoding_type)

    # 1. VERİ BİRLEŞTİRME (ZİNCİRLEME MERGE)
    print("Veriler birleştiriliyor...")
    
    # Her rota için örnek BİR sefer alıyoruz
    representative_trips = trips.groupby('route_id').first().reset_index()[['route_id', 'trip_id', 'trip_headsign']]
    
    # Rota bilgisini ekle
    route_trips = pd.merge(routes[['route_id', 'route_short_name', 'route_long_name']], representative_trips, on='route_id')
    
    # Durak zamanlarını ve sıralamayı ekle
    trip_details = pd.merge(route_trips, stop_times[['trip_id', 'stop_id', 'stop_sequence']], on='trip_id')
    
    # Durak koordinatlarını ekle
    full_df = pd.merge(trip_details, stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], on='stop_id')

    # 2. SIRALAMA
    full_df = full_df.sort_values(by=['route_id', 'stop_sequence'])

    # 3. MESAFE VE SÜRE HESABI
    print("Mesafeler hesaplanıyor...")

    full_df['next_stop_name'] = full_df.groupby('route_id')['stop_name'].shift(-1)
    full_df['next_lat'] = full_df.groupby('route_id')['stop_lat'].shift(-1)
    full_df['next_lon'] = full_df.groupby('route_id')['stop_lon'].shift(-1)

    # Haversine Formülü
    def haversine_vec(lat1, lon1, lat2, lon2):
        R = 6371000 # Metre
        phi1, phi2 = np.radians(lat1), np.radians(lat2)
        dphi = np.radians(lat2 - lat1)
        dlambda = np.radians(lon2 - lon1)
        a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
        return R * c

    full_df['dist_meters'] = haversine_vec(full_df['stop_lat'], full_df['stop_lon'], full_df['next_lat'], full_df['next_lon'])

    # Hız Tanımları (Metrobüs 40, Diğerleri 25 km/s)
    # route_short_name bazen sayı (34) bazen string olabilir, str() ile garantiye alıyoruz
    full_df['avg_speed_kmh'] = full_df['route_short_name'].apply(lambda x: 40 if str(x).startswith('34') else 25)
    
    # Süre (Dakika)
    full_df['duration_min'] = (full_df['dist_meters'] / (full_df['avg_speed_kmh'] / 3.6)) / 60

    # Temizlik
    full_df = full_df.dropna(subset=['dist_meters'])

    return full_df[['route_short_name', 'stop_name', 'next_stop_name', 'dist_meters', 'duration_min']]

# --- KULLANIM ---
sonuc = process_transport_data('stops.csv', 'routes.csv', 'trips.csv', 'stop_times.csv')
sonuc.to_csv('processed_routes.csv', index=False)
print(sonuc.head())

Dosyalar okunuyor...


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xfd in position 163: invalid start byte